<a href="https://colab.research.google.com/github/onyourwon/colab/blob/main/W13_llm_workflow_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W12 · LLM 심화 통합 실습 — 근거, 구조화, 검증, 비교

오늘 노트북은 **하나의 파일 안에 여러 단계**로 구성됩니다.

1. 기본 예제: naive → grounded → structured → validation
2. 다중 LLM 비교: Gemini / ChatGPT / Claude
3. 추가 self-practice 예제 bank

> 목표: **그럴듯한 답**이 아니라 **근거가 있고, 다시 쓸 수 있고, 다시 확인된 답**을 만드는 것

## 오늘 핵심 개념 4개

### 1) Grounding = 근거 붙이기
- 내가 제공한 문서, 메모, 리뷰를 먼저 주고 답하게 하기

### 2) Structured output = 구조화 출력
- 자유문 대신 JSON/표/체크리스트로 결과 받기

### 3) Validation loop = 검증 루프
- 첫 답을 그대로 믿지 않고 다시 확인하기

### 4) Multi-LLM comparison = 모델 비교
- 같은 프롬프트라도 LLM마다 형식 준수, 과신, 근거 처리 방식이 다를 수 있음

---
## 섹션 1. 기본 실습 데이터 준비

지난주 상권분석 흐름을 이어서, **스터디카페 입지 후보를 판단하는 메모 8개**를 사용합니다.

In [ ]:
import json
import re
import pandas as pd

docs = pd.DataFrame([
    {"doc_id": "D1", "kind": "현장메모", "text": "A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이 부족하다는 말이 있었다."},
    {"doc_id": "D2", "kind": "임대료메모", "text": "A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이 이미 있다."},
    {"doc_id": "D3", "kind": "현장메모", "text": "B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 유동인구가 급격히 줄어든다."},
    {"doc_id": "D4", "kind": "경쟁메모", "text": "B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카페는 확인되지 않았다."},
    {"doc_id": "D5", "kind": "현장메모", "text": "C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이 크고 회전이 빠르다."},
    {"doc_id": "D6", "kind": "학생설문", "text": "학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게 꼽았다."},
    {"doc_id": "D7", "kind": "운영규정", "text": "밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지 않다."},
    {"doc_id": "D8", "kind": "인터뷰", "text": "시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳이 없다는 의견이 많았다."},
])

docs

,doc_id,kind,text
0,D1,현장메모,A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이...
1,D2,임대료메모,A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이...
2,D3,현장메모,B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 ...
3,D4,경쟁메모,"B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카..."
4,D5,현장메모,C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이...
5,D6,학생설문,"학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게 ..."
6,D7,운영규정,밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지...
7,D8,인터뷰,시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳...


In [ ]:
question = "학교 앞에 20석 규모 스터디카페를 열 때 가장 적합한 후보 구역 1곳을 고르고 이유와 위험을 설명해줘."
print(question)

학교 앞에 20석 규모 스터디카페를 열 때 가장 적합한 후보 구역 1곳을 고르고 이유와 위험을 설명해줘.


In [ ]:
context_text = "\n".join([f"[{row.doc_id}] ({row.kind}) {row.text}" for _, row in docs.iterrows()])
print(context_text)

[D1] (현장메모) A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이 부족하다는 말이 있었다.
[D2] (임대료메모) A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이 이미 있다.
[D3] (현장메모) B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 유동인구가 급격히 줄어든다.
[D4] (경쟁메모) B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카페는 확인되지 않았다.
[D5] (현장메모) C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이 크고 회전이 빠르다.
[D6] (학생설문) 학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게 꼽았다.
[D7] (운영규정) 밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지 않다.
[D8] (인터뷰) 시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳이 없다는 의견이 많았다.


---
## 섹션 2. naive 답변 vs grounded 답변

같은 질문을 두 번 던져 차이를 봅니다.

In [ ]:
naive_prompt = f'''
너는 친절한 분석 도우미야.

질문:
{question}
'''

print(naive_prompt)


너는 친절한 분석 도우미야.

질문:
학교 앞에 20석 규모 스터디카페를 열 때 가장 적합한 후보 구역 1곳을 고르고 이유와 위험을 설명해줘.



In [ ]:
naive_answer = '''
소형 스터디카페 창업을 고민 중이시군요! 20석 규모는 공간이 아늑한 만큼, 대형 프랜차이즈와는 완전히 다른 전략으로 접근해야 성공 확률을 높일 수 있습니다.

친절하고 꼼꼼하게 분석해 드릴게요. 😉

---

## 📍 가장 적합한 후보 구역: 대학가 후문 근처의 원룸·자취방 밀집 지역 (이면도로)

대로변이나 정문 앞 중심 상권이 아니라, **학생들이 실제로 거주하며 이동하는 후문 쪽 골목이나 자취방 초입**을 추천합니다.

### 💡 선정 이유 (왜 이곳일까요?)

* **치명적인 임대료 절감 (고정비 최소화):** 20석은 만석이 되어도 하루 매출의 한계(상한선)가 명확합니다. 따라서 중심가의 비싼 임대료를 감당하면 적자를 보기 쉽습니다. 임대료가 비교적 저렴한 골목 상권으로 들어가야 안전합니다.
* **‘슬세권’ 공략 (압도적인 접근성):** 대학생들은 시험 기간이나 늦은 밤에 슬리퍼를 신고 편하게 갈 수 있는 집 앞 공간을 선호합니다. 학교 도서관이 문을 닫거나 자취방에서 집중이 안 될 때, 바로 옆에 있는 소규모 스터디카페는 최고의 선택지가 됩니다.
* **대형 브랜드와의 정면 대결 회피:** 60~100석 규모의 대형 프랜차이즈들은 유동 인구가 많은 대로변에 주로 입점합니다. 자취방 밀집 구역 깊숙이 들어가면, 대형 매장들이 커버하지 못하는 '동네 단골'을 독점할 수 있습니다.

---

## ⚠️ 이 구역의 위험 요소 (꼭 알아두셔야 해요!)

**1. 대학가 특유의 극심한 '방학 비수기'**

* **위험:** 여름과 겨울방학(일 년 중 약 4~5개월) 동안은 자취하던 학생들이 대거 본가로 내려갑니다. 이 시기에는 유동 인구가 눈에 띄게 줄어 매출이 급감할 수 있습니다.
* **대비책:** 학기 중에 고정 회원(기간권)을 최대한 확보해 두고, 방학 시즌에는 인근 고등학생이나 취업 준비생, 자격증을 준비하는 직장인을 타겟으로 한 할인 이벤트를 적극적으로 펼쳐야 합니다.

**2. 좌석 수 한계로 인한 매출 정체**

* **위험:** 시험 기간에 손님이 아무리 몰려도 20석이 꽉 차면 더 이상 수익을 낼 수 없습니다. 반면, 한 사람이 음료 하나만 시켜놓고 온종일 자리를 차지하는 '카공족'이 많아지면 회전율이 떨어져 매출에 타격을 입습니다.
* **대비책:** 무제한 이용권보다는 '시간 요금제'나 '시간 충전권' 위주로 요금 체계를 짜서 자연스러운 좌석 회전을 유도하는 것이 좋습니다.

**3. 인근 대형 매장의 기습 입점**

* **위험:** 만약 근처에 조금 더 큰 규모의 세련된 스터디카페가 새로 들어온다면, 휴게실이나 편의시설(프린터, 간식 바 등) 경쟁에서 밀려 회원을 뺏길 위험이 있습니다.
* **대비책:** 규모로 경쟁하기보다 '점주가 직접 관리하는 철저한 면학 분위기'와 '조용하고 아늑한 독서실 같은 감성'을 무기로 삼아야 합니다. 소형 매장일수록 백색소음 관리, 청결 상태, 민원 해결 속도에서 강점을 가질 수 있습니다.

---

> **📌 핵심 한 줄 요약**
> 20석 규모는 **"무인 키오스크를 통한 인건비 절감 + 저렴한 후문 임대료"** 조합으로 고정 지출을 완전히 쥐어짜고, 자취생들의 '집 앞 아지트'가 되는 전략이 가장 안전합니다.
'''

print(naive_answer[:400])

In [ ]:
grounded_prompt = f'''
너는 친절한 분석 도우미야.
아래 문서 묶음만 근거로 사용해서 답해줘.
답변에는 근거가 되는 문서 ID를 반드시 괄호로 붙여줘.
문서에 없는 내용은 추측하지 말고, 모르면 모른다고 말해줘.

문서 묶음:
[D1] (현장메모) A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이 부족하다는 말이 있었다.
[D2] (임대료메모) A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이 이미 있다.
[D3] (현장메모) B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 유동인구가 급격히 줄어든다.
[D4] (경쟁메모) B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카페는 확인되지 않았다.
[D5] (현장메모) C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이 크고 회전이 빠르다.
[D6] (학생설문) 학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게 꼽았다.
[D7] (운영규정) 밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지 않다.
[D8] (인터뷰) 시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳이 없다는 의견이 많았다.

질문:
학교 앞에 20석 규모 스터디카페를 열 때 가장 적합한 후보 구역 1곳을 고르고 이유와 위험을 설명해줘.
'''

print(grounded_prompt)

In [ ]:
grounded_answer = '''
제공해주신 문서들을 바탕으로 20석 규모의 스터디카페를 열기에 가장 적합한 후보 구역 1곳과 그 이유, 그리고 위험 요소를 분석해 드립니다.

---

### 📍 가장 적합한 후보 구역: B구역

### 💡 선정 이유

1. **학생들의 핵심 니즈 충족:** 학생들은 스터디카페를 선택할 때 '조용한 분위기'와 '2시간 이상 머물 수 있는 좌석'을 가장 중요하게 생각합니다 [D6]. B구역은 비교적 조용한 환경을 갖추고 있으며 [D3], 주변에 오래 머물 수 있는 스터디 좌석형 카페가 없어 [D4] 이러한 수요를 독점하기에 유리합니다.
2. **소규모 운영에 적합한 저렴한 고정비:** 20석 규모의 소형 매장 특성상 고정비 관리가 중요한데, B구역은 월세가 130만원 수준으로 저렴하여 [D3] 월세가 210만원 수준으로 높은 A구역 [D2]에 비해 비용 부담이 적습니다.
3. **낮은 경쟁 강도:** 주변에 프랜차이즈 카페 1곳만 존재하여 [D4], 이미 테이크아웃 카페가 4곳이나 있는 A구역 [D2]이나 소음이 심하고 회전이 빠른 C구역 [D5]에 비해 경쟁 환경이 양호합니다.

### ⚠️ 위험 요소

* **야간 유동인구 급감으로 인한 운영 제약:** B구역은 밤 8시 이후 유동인구가 급격히 줄어드는 특성이 있습니다 [D3]. 이로 인해 늦은 저녁 시간대의 이용률과 매출이 떨어질 수 있는 위험이 존재합니다.
'''

print(grounded_answer[:500])

#### 비교 메모
- naive 답변의 한계:
- grounded 답변의 장점:
- grounded 답변에도 아직 부족한 점:

---
## 섹션 3. 구조화 출력 — JSON으로 받기

In [ ]:
json_schema = {
    "candidate_zone": "A구역/B구역/C구역 중 하나",
    "fit_reason": "한 줄 설명",
    "evidence_ids": ["D1", "D6"],
    "risk": "위험 한 줄",
    "confidence_1_to_5": 3,
    "missing_info": "추가로 필요하지만 현재 없는 정보"
}

print(json.dumps(json_schema, ensure_ascii=False, indent=2))

In [ ]:
structured_prompt = f'''
너는 친절한 분석 도우미야.
아래 문서 묶음만 근거로 사용해서 답해줘.
문서에 없는 내용은 추측하지 말고 missing_info에 적어줘.
반드시 JSON만 출력해줘. 설명 문장은 쓰지 마.

문서 묶음:
[D1] (현장메모) A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이 부족하다는 말이 있었다.
[D2] (임대료메모) A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이 이미 있다.
[D3] (현장메모) B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 유동인구가 급격히 줄어든다.
[D4] (경쟁메모) B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카페는 확인되지 않았다.
[D5] (현장메모) C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이 크고 회전이 빠르다.
[D6] (학생설문) 학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게 꼽았다.
[D7] (운영규정) 밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지 않다.
[D8] (인터뷰) 시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳이 없다는 의견이 많았다.

질문:
학교 앞에 20석 규모 스터디카페를 열 때 가장 적합한 후보 구역 1곳을 고르고 이유와 위험을 설명해줘.

JSON 형식:
{json.dumps(json_schema, ensure_ascii=False, indent=2)}
'''

print(structured_prompt)

In [ ]:
json_text = r'''
{
"candidate_zone": "B구역",
"fit_reason": "B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하며, 주변에 오래 머물 수 있는 스터디 좌석형 카페가 확인되지 않아 학생들이 가장 중요하게 꼽은 조용한 분위기와 2시간 이상 머물 수 있는 좌석이라는 니즈에 적합합니다.",
"evidence_ids": [
"D3",
"D4",
"D6"
],
"risk": "밤 8시 이후 유동인구가 급격히 줄어든다는 점입니다.",
"confidence_1_to_5": 5,
"missing_info": "제공된 문서에 '20석 규모'의 스터디카페를 창업할 때 구체적으로 어떤 구역이 정량적·정성적으로 '가장 적합한지' 직접적으로 지정하여 명시한 결론 데이터는 없습니다. 따라서 제공된 구역별 특징(월세, 소음, 경쟁자 등)과 학생들의 요구사항을 바탕으로 논리적으로 추론하여 선택했습니다."
}
'''

print(json_text)

In [ ]:
parsed = json.loads(json_text)
pd.DataFrame([parsed])

#### 구조화 출력 메모
- 자유문보다 좋은 점:
- 여전히 조심해야 할 점:

---
## 섹션 4. 검증 루프

In [ ]:
allowed_doc_ids = set(docs["doc_id"])
allowed_zones = {"A구역", "B구역", "C구역"}

def validate_answer(obj):
    problems = []
    required_keys = {
        "candidate_zone",
        "fit_reason",
        "evidence_ids",
        "risk",
        "confidence_1_to_5",
        "missing_info",
    }

    missing = required_keys - set(obj.keys())
    if missing:
        problems.append(f"키 누락: {sorted(missing)}")

    zone = obj.get("candidate_zone")
    if zone not in allowed_zones:
        problems.append(f"candidate_zone 값 이상: {zone}")

    evidence_ids = obj.get("evidence_ids", [])
    if not isinstance(evidence_ids, list):
        problems.append("evidence_ids는 리스트여야 함")
    else:
        bad_ids = [x for x in evidence_ids if x not in allowed_doc_ids]
        if bad_ids:
            problems.append(f"존재하지 않는 evidence_ids: {bad_ids}")

    confidence = obj.get("confidence_1_to_5")
    if not isinstance(confidence, int) or not (1 <= confidence <= 5):
        problems.append(f"confidence_1_to_5 값 이상: {confidence}")

    if len(str(obj.get("fit_reason", "")).strip()) < 10:
        problems.append("fit_reason이 너무 짧음")

    if len(str(obj.get("risk", "")).strip()) < 5:
        problems.append("risk가 너무 짧음")

    return problems

problems = validate_answer(parsed)
if problems:
    print("형식/검증 이슈 발견")
    for p in problems:
        print("-", p)
else:
    print("기본 형식 검증 통과")

In [ ]:
review_prompt = f'''
너는 엄격한 검토자야.
아래 문서 묶음과 JSON 답변을 보고,
1) 근거가 약한 부분 2개
2) 과신한 부분 1개
3) 추가로 필요한 정보 2개
를 짧게 적어줘.

문서 묶음:
[D1] (현장메모) A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이 부족하다는 말이 있었다.
[D2] (임대료메모) A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이 이미 있다.
[D3] (현장메모) B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 유동인구가 급격히 줄어든다.
[D4] (경쟁메모) B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카페는 확인되지 않았다.
[D5] (현장메모) C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이 크고 회전이 빠르다.
[D6] (학생설문) 학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게 꼽았다.
[D7] (운영규정) 밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지 않다.
[D8] (인터뷰) 시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳이 없다는 의견이 많았다.

JSON 답변:
{
"candidate_zone": "B구역",
"fit_reason": "B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하며, 주변에 오래 머물 수 있는 스터디 좌석형 카페가 확인되지 않아 학생들이 가장 중요하게 꼽은 조용한 분위기와 2시간 이상 머물 수 있는 좌석이라는 니즈에 적합합니다.",
"evidence_ids": [
"D3",
"D4",
"D6"
],
"risk": "밤 8시 이후 유동인구가 급격히 줄어든다는 점입니다.",
"confidence_1_to_5": 5,
"missing_info": "제공된 문서에 '20석 규모'의 스터디카페를 창업할 때 구체적으로 어떤 구역이 정량적·정성적으로 '가장 적합한지' 직접적으로 지정하여 명시한 결론 데이터는 없습니다. 따라서 제공된 구역별 특징(월세, 소음, 경쟁자 등)과 학생들의 요구사항을 바탕으로 논리적으로 추론하여 선택했습니다."
}
'''

print(review_prompt)

#### 검토 결과
- 근거가 약한 부분:
- 과신한 부분:
- 추가로 필요한 정보:

In [ ]:
revised_json_text = r'''
엄격한 검토자 관점에서 해당 JSON 답변을 분석한 결과입니다.

---

### 1) 근거가 약한 부분 (2개)

* **B구역의 '2시간 이상 머물 수 있는 좌석' 니즈 충족 판단:** [D4]는 B구역에 스터디 좌석형 카페가 '확인되지 않았다'는 경쟁 현황일 뿐입니다. 답변은 이를 바탕으로 B구역이 학생들이 원하는 '2시간 이상 머물 수 있는 좌석'을 제공하기에 적합하다고 결론지었으나, 이는 공급 공백을 의미할 뿐 B구역 자체가 해당 좌석을 구성하기에 최적이라는 직접적인 근거가 되기엔 약합니다.
* **A구역(도서관 인근)의 핵심 수요 배제:** [D1]에서 A구역은 도서관과 3분 거리이며 '학생이 1~2시간 머물 공간이 부족하다'고 명시했고, [D8]에서도 '시험기간 도서관 주변 좌석 부족 민원'을 언급했습니다. 20석 규모의 소형 스터디카페라면 회전율과 밀착 수요가 중요한데, A구역의 명확한 수요 데이터를 단지 월세가 높다는 이유로 깊이 있게 비교하지 않고 B구역을 단정 지은 것은 근거가 부족합니다.

### 2) 과신한 부분 (1개)

* **신뢰도 점수 `confidence_1_to_5`: 5**
* `missing_info`에 본인 스스로 '20석 규모 창업에 어떤 구역이 가장 적합한지 지정된 결론 데이터가 없다'고 명시하며 논리적 추론(추측)을 개입시켰음을 인정했습니다. 조건에 따른 명확한 정량적 최적화 근거가 없는 상태에서 신뢰도를 만점(5점)으로 부여한 것은 명백한 과신입니다.



### 3) 추가로 필요한 정보 (2개)

* **20석 규모 운영 시 구역별 예상 손익분기점(BEP) 데이터:** A구역의 높은 월세(210만 원)와 B구역의 낮은 월세(130만 원)가 매출 상한선이 명확한 '20석 규모'의 스터디카페 수익성에 각각 어떤 영향을 미치는지 판단할 수 있는 좌석당 기대 매출 정보가 필요합니다.
* **B구역의 밤 8시 이후 유동인구 감소가 주 타겟(학생) 이용에 미치는 영향:** 스터디카페는 보통 야간 이용률이 높습니다. B구역의 밤 8시 이후 유동인구 급감[D3]이 실제 학생들의 하교/귀가로 인한 것인지, 혹은 단순히 상권 자체의 조기 마감 때문인지에 대한 구체적인 인과관계 정보가 필요합니다.
'''

revised = json.loads(revised_json_text)
pd.DataFrame([revised])

In [ ]:
revised_problems = validate_answer(revised)
if revised_problems:
    print("수정본에도 이슈가 남아 있음")
    for p in revised_problems:
        print("-", p)
else:
    print("수정본 형식 검증 통과")

---
## 섹션 5. 다중 LLM 비교 실험

같은 프롬프트를 **Gemini / ChatGPT / Claude**에 던져 결과 차이를 봅니다.

> 3개가 어렵다면 2개만 비교해도 됩니다.

In [ ]:
model_answers = {
    "Gemini": '''제공해주신 문서들을 바탕으로 20석 규모의 스터디카페를 열기에 가장 적합한 후보 구역 1곳과 그 이유, 그리고 위험 요소를 분석해 드립니다.

---

### 📍 가장 적합한 후보 구역: B구역

### 💡 선정 이유

1. **학생들의 핵심 니즈 충족:** 학생들은 스터디카페를 선택할 때 '조용한 분위기'와 '2시간 이상 머물 수 있는 좌석'을 가장 중요하게 생각합니다 [D6]. B구역은 비교적 조용한 환경을 갖추고 있으며 [D3], 주변에 오래 머물 수 있는 스터디 좌석형 카페가 없어 [D4] 이러한 수요를 독점하기에 유리합니다.
2. **소규모 운영에 적합한 저렴한 고정비:** 20석 규모의 소형 매장 특성상 고정비 관리가 중요한데, B구역은 월세가 130만원 수준으로 저렴하여 [D3] 월세가 210만원 수준으로 높은 A구역 [D2]에 비해 비용 부담이 적습니다.
3. **낮은 경쟁 강도:** 주변에 프랜차이즈 카페 1곳만 존재하여 [D4], 이미 테이크아웃 카페가 4곳이나 있는 A구역 [D2]이나 소음이 심하고 회전이 빠른 C구역 [D5]에 비해 경쟁 환경이 양호합니다.

### ⚠️ 위험 요소

* **야간 유동인구 급감으로 인한 운영 제약:** B구역은 밤 8시 이후 유동인구가 급격히 줄어드는 특성이 있습니다 [D3]. 이로 인해 늦은 저녁 시간대의 이용률과 매출이 떨어질 수 있는 위험이 존재합니다.''',
    "ChatGPT": '''가장 적합한 후보 구역은 **B구역**이라고 판단됩니다.

**선정 이유**

1. **스터디카페 핵심 수요와 맞는 환경**

* 학생들은 조용한 분위기와 2시간 이상 머물 수 있는 좌석을 가장 중요하게 꼽았습니다. (D6)
* B구역은 비교적 조용한 환경으로 설명되어 있어 이러한 수요와 잘 맞습니다. (D3)
* 주변에 오래 머물 수 있는 스터디 좌석형 카페가 확인되지 않아 차별화 가능성이 있습니다. (D4)

2. **경쟁 강도가 상대적으로 낮음**

* B구역 주변에는 프랜차이즈 카페 1곳만 있습니다. (D4)
* 반면 A구역은 반경 200m 내 테이크아웃 카페가 이미 4곳 있습니다. (D2)
* 따라서 B구역은 경쟁 부담이 상대적으로 낮을 가능성이 있습니다. (D2, D4)

3. **임대료 부담이 낮음**

* B구역 월세는 130만원 수준으로 저렴합니다. (D3)
* A구역은 월세가 210만원 수준으로 높습니다. (D2)
* 20석 규모 스터디카페 운영에서 임대료 부담은 중요한 요소이므로 B구역이 유리합니다. (D2, D3)

**B구역의 위험 요소**

* 가장 큰 위험은 **저녁 시간 수요 감소**입니다.
* B구역은 밤 8시 이후 유동인구가 급격히 줄어듭니다. (D3)
* 또한 늦은 시간 행사형 운영은 운영규정상 적합하지 않습니다. (D7)
* 따라서 야간 유동인구나 행사 운영에 의존하는 방식은 어려울 수 있습니다. (D3, D7)

**다른 구역과 비교한 이유**

* **A구역**은 도서관과 가깝고 학생 체류 공간 부족 수요가 있으며 시험기간 좌석 부족 문제도 확인됩니다. (D1, D8) 다만 월세가 높고 경쟁 카페가 많습니다. (D2)
* **C구역**은 유동인구는 매우 많지만 소음이 크고 회전이 빠른 환경이라 학생들이 선호한 조용하고 오래 머무는 공간과는 다소 맞지 않습니다. (D5, D6)

따라서 **조용한 장기 체류형 스터디카페라는 콘셉트와 비용·경쟁 환경을 종합하면 B구역이 가장 적합한 후보**로 보입니다. 다만 **밤 8시 이후 유동인구 감소가 주요 운영 위험**입니다. (D3, D4, D6)
''',
    "Claude": '''**추천 구역: B구역**

**추천 이유**

학생 수요와 가장 잘 맞습니다. 학생들이 가장 중요하게 꼽은 조건이 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석인데 (D6), B구역은 비교적 조용하고 (D3) 주변에 오래 머물 수 있는 스터디 좌석형 카페가 아직 없어 (D4) 공백 시장을 노릴 수 있습니다. 월세도 130만원으로 A구역(210만원)보다 80만원 저렴해 초기 운영 부담이 낮습니다 (D2, D3). 시험기간마다 도서관 주변 좌석 부족 민원이 반복된다는 점도 수요 근거가 됩니다 (D8).

**A구역, C구역을 고르지 않은 이유**

A구역은 유동인구와 도서관 접근성은 좋지만 (D1), 월세가 높고 테이크아웃 카페가 이미 4곳이나 있어 경쟁이 치열합니다 (D2). C구역은 유동인구가 많지만 소음이 크고 회전이 빠른 상권이라 (D5) 조용한 분위기와 장시간 체류를 원하는 스터디카페 콘셉트와 맞지 않습니다 (D6).

**B구역의 위험 요소**

밤 8시 이후 유동인구가 급격히 줄어드는 점이 가장 큰 리스크입니다 (D3). 저녁 시간대 매출이 낮을 수 있고, 운영규정상 밤 10시 이후 소음 제한도 있어 (D7) 야간 운영 여지가 제한됩니다. 또한 B구역의 낮 시간대 유동인구 수준, 주변 거주 학생 수, 실제 접근성 등은 제공된 문서에 없어 추가 현장 조사가 필요합니다.''',
}

comparison_rows = []
for model_name, text in model_answers.items():
    evidence_ids = sorted(set(re.findall(r"D\d+", text)))
    comparison_rows.append({
        "model": model_name,
        "char_len": len(text.strip()),
        "evidence_count": len(evidence_ids),
        "evidence_ids": ", ".join(evidence_ids),
    })

pd.DataFrame(comparison_rows)

#### grounded 답변 모델 비교 메모
- 어떤 모델이 근거를 가장 잘 붙였나:
- 어떤 모델이 가장 장황했나:
- 어떤 모델이 가장 과신했나:

In [ ]:
model_json_answers = {
    "Gemini": r'''{
"candidate_zone": "B구역",
"fit_reason": "B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하며, 주변에 오래 머물 수 있는 스터디 좌석형 카페가 확인되지 않아 학생들이 가장 중요하게 꼽은 조용한 분위기와 2시간 이상 머물 수 있는 좌석이라는 니즈에 적합합니다.",
"evidence_ids": [
"D3",
"D4",
"D6"
],
"risk": "밤 8시 이후 유동인구가 급격히 줄어든다는 점입니다.",
"confidence_1_to_5": 5,
"missing_info": "제공된 문서에 '20석 규모'의 스터디카페를 창업할 때 구체적으로 어떤 구역이 정량적·정성적으로 '가장 적합한지' 직접적으로 지정하여 명시한 결론 데이터는 없습니다. 따라서 제공된 구역별 특징(월세, 소음, 경쟁자 등)과 학생들의 요구사항을 바탕으로 논리적으로 추론하여 선택했습니다."
}''',
    "ChatGPT": r'''{
"candidate_zone": "B구역",
"fit_reason": "B구역은 비교적 조용하고 월세가 저렴하며, 주변에 프랜차이즈 카페 1곳만 있고 오래 머물 수 있는 스터디 좌석형 카페가 확인되지 않았다. 학생들은 조용한 분위기와 2시간 이상 머물 수 있는 좌석을 중요하게 꼽았다.",
"evidence_ids": ["D3", "D4", "D6"],
"risk": "밤 8시 이후 유동인구가 급격히 줄어든다. 또한 늦은 시간 행사형 운영은 적합하지 않다.",
"confidence_1_to_5": 4,
"missing_info": "20석 규모 운영에 필요한 실제 수요 규모, 구역별 콘센트·공간 확보 가능성, 매출 전망, 운영시간별 이용 수요, C구역과 B구역의 학생 비중, A구역·C구역의 스터디카페 경쟁 현황은 문서에 없다."
}
''',
    "Claude": r'''```json
{
  "candidate_zone": "B구역",
  "fit_reason": "학생 수요(조용한 분위기, 장시간 체류)와 가장 잘 부합하며, 월세가 130만원으로 저렴하고 스터디 좌석형 경쟁업체가 없는 공백 시장이다.",
  "evidence_ids": ["D3", "D4", "D6", "D8"],
  "risk": "밤 8시 이후 유동인구가 급격히 줄어 저녁 매출이 낮을 수 있고, 운영규정상 야간 행사 운영도 제한된다.",
  "confidence_1_to_5": 3,
  "missing_info": "B구역의 낮 시간대 유동인구 규모, 인근 학생 거주 밀도, 도보 접근성, 주차 여건이 문서에 없어 추가 현장 조사가 필요하다."
}
```''',
}

def try_parse_json(text):
    try:
        obj = json.loads(text)
        return True, obj
    except Exception as e:
        return False, str(e)

rows = []
for model_name, text in model_json_answers.items():
    ok, result = try_parse_json(text)
    row = {"model": model_name, "json_parse_ok": ok}
    if ok:
        row["keys"] = ", ".join(sorted(result.keys()))
        row["validation_issues"] = " / ".join(validate_answer(result)) or "통과"
    else:
        row["keys"] = "-"
        row["validation_issues"] = str(result)
    rows.append(row)

pd.DataFrame(rows)

#### structured 출력 모델 비교 메모
- 어떤 모델이 JSON 규칙을 가장 잘 지켰나:
- 어떤 모델이 missing_info를 가장 솔직하게 썼나:
- 어떤 모델 결과를 실무에 다시 쓰기 가장 쉬웠나:

---
## 섹션 6. 추가 self-practice 예제 bank

시간이 남으면 아래 예제 중 1개를 골라 같은 흐름으로 다시 해 보세요.

In [ ]:
challenge_cases = {
    "review_summary": {
        "title": "카페 리뷰 요약기",
        "question": "리뷰 묶음을 보고 장점 3개, 불만 3개, 가장 먼저 고칠 점 1개를 정리해줘.",
        "schema": {
            "top_strengths": [],
            "top_issues": [],
            "quick_fix": "",
            "evidence_ids": [],
            "confidence_1_to_5": 0,
        },
        "docs": [
            {"doc_id": "R1", "kind": "리뷰", "text": "좌석이 편하고 콘센트가 많아서 좋았다."},
            {"doc_id": "R2", "kind": "리뷰", "text": "음악 소리가 커서 오래 공부하기 어렵다."},
            {"doc_id": "R3", "kind": "리뷰", "text": "직원이 친절하고 음료가 빨리 나왔다."},
            {"doc_id": "R4", "kind": "리뷰", "text": "화장실 청결이 아쉬웠다."},
            {"doc_id": "R5", "kind": "리뷰", "text": "시험기간에 자리가 부족했다."},
            {"doc_id": "R6", "kind": "리뷰", "text": "디저트는 평범하지만 공간은 조용했다."},
        ],
    },
    "survey_sort": {
        "title": "자유응답 설문 분류기",
        "question": "설문 응답을 칭찬/불만/개선요청으로 분류하고, 가장 우선순위 높은 개선요청 2개를 골라줘.",
        "schema": {
            "top_priority_requests": [],
            "categories": [],
            "evidence_ids": [],
            "confidence_1_to_5": 0,
        },
        "docs": [
            {"doc_id": "S1", "kind": "설문", "text": "실습은 재미있었지만 속도가 조금 빨랐다."},
            {"doc_id": "S2", "kind": "설문", "text": "예제가 많아서 좋았다."},
            {"doc_id": "S3", "kind": "설문", "text": "과제가 헷갈려서 제출 예시가 더 필요하다."},
            {"doc_id": "S4", "kind": "설문", "text": "교수님 설명이 차분해서 좋았다."},
            {"doc_id": "S5", "kind": "설문", "text": "용어 설명을 더 쉽게 해주면 좋겠다."},
            {"doc_id": "S6", "kind": "설문", "text": "실습 시간이 더 길었으면 좋겠다."},
        ],
    },
    "team_roles": {
        "title": "팀플 역할 배치 도우미",
        "question": "팀원 4명에게 발표/자료조사/디자인/일정관리 역할을 배정하고, 충돌 위험이 있으면 함께 적어줘.",
        "schema": {
            "assignments": [],
            "risk": "",
            "evidence_ids": [],
            "confidence_1_to_5": 0,
        },
        "docs": [
            {"doc_id": "T1", "kind": "자기소개", "text": "민수는 발표 경험이 많고 사람 앞에서 말하는 걸 덜 두려워한다."},
            {"doc_id": "T2", "kind": "자기소개", "text": "지연은 꼼꼼하고 일정표 정리를 잘한다."},
            {"doc_id": "T3", "kind": "자기소개", "text": "하준은 포토샵과 PPT 디자인 경험이 있다."},
            {"doc_id": "T4", "kind": "자기소개", "text": "서윤은 자료 찾는 속도가 빠르고 문서 정리를 잘한다."},
        ],
    },
    "rule_checklist": {
        "title": "규정 체크리스트 만들기",
        "question": "규정을 해야 할 것/하면 안 되는 것/확인 필요한 것으로 나누고, 신입생이 실수하기 쉬운 항목 3개를 골라줘.",
        "schema": {
            "must_do": [],
            "must_not_do": [],
            "needs_confirmation": [],
            "risk_points": [],
            "evidence_ids": [],
        },
        "docs": [
            {"doc_id": "P1", "kind": "규정", "text": "동아리실 사용 후 전등과 냉난방기를 반드시 끈다."},
            {"doc_id": "P2", "kind": "규정", "text": "외부인 출입은 사전 승인 없이 허용되지 않는다."},
            {"doc_id": "P3", "kind": "규정", "text": "음식물 반입은 가능하나 냄새가 강한 음식은 자제한다."},
            {"doc_id": "P4", "kind": "규정", "text": "기자재 대여는 담당자 확인 후 기록지 작성이 필요하다."},
        ],
    },
}

pd.DataFrame([
    {"case_key": k, "title": v["title"], "doc_count": len(v["docs"]), "question": v["question"]}
    for k, v in challenge_cases.items()
])

In [ ]:
selected_case = "review_summary"   # 다른 예제를 하려면 여기 값을 바꾸세요.

case = challenge_cases[selected_case]
case_docs = pd.DataFrame(case["docs"])
case_question = case["question"]
case_schema = case["schema"]
case_context_text = "\n".join([f"[{row.doc_id}] ({row.kind}) {row.text}" for _, row in case_docs.iterrows()])

print("축구 동아리 규정 체크리스트 만들기:", case["title"])
print(case_question)
case_docs

In [ ]:
case_grounded_prompt = f'''
너는 친절한 분석 도우미야.
아래 문서 묶음만 근거로 사용해서 답해줘.
답변에는 근거 ID를 붙여줘.
문서에 없는 내용은 추측하지 말고 모르면 모른다고 말해줘.

문서 묶음:
 {"doc_id": "S1", "kind": "규정", "text": "경기 참가 전 반드시 준비운동을 10분 이상 실시한다."},
        {"doc_id": "S2", "kind": "규정", "text": "유니폼과 축구화는 개인이 준비하며, 공용 조끼는 사용 후 제자리에 반납한다."},
        {"doc_id": "S3", "kind": "규정", "text": "구장 예약은 총무가 담당하며, 개인이 임의로 변경하거나 취소할 수 없다."},
        {"doc_id": "S4", "kind": "규정", "text": "부상 발생 시 즉시 활동을 중단하고 동아리 회장에게 보고한다."},
        {"doc_id": "S5", "kind": "규정", "text": "외부 팀과의 친선경기는 부장 승인 후 최소 1주일 전에 공지해야 한다."},
        {"doc_id": "S6", "kind": "규정", "text": "훈련 불참 시 24시간 전에 단체 채팅방에 사전 공지를 남긴다."},

질문:
가장 중요한 규정 하나 골라줘
'''

print(case_grounded_prompt)

In [ ]:
case_structured_prompt = f'''
너는 친절한 분석 도우미야.
아래 문서 묶음만 근거로 사용해서 답해줘.
반드시 JSON만 출력해줘.

문서 묶음:
 {"doc_id": "S1", "kind": "규정", "text": "경기 참가 전 반드시 준비운동을 10분 이상 실시한다."},
        {"doc_id": "S2", "kind": "규정", "text": "유니폼과 축구화는 개인이 준비하며, 공용 조끼는 사용 후 제자리에 반납한다."},
        {"doc_id": "S3", "kind": "규정", "text": "구장 예약은 총무가 담당하며, 개인이 임의로 변경하거나 취소할 수 없다."},
        {"doc_id": "S4", "kind": "규정", "text": "부상 발생 시 즉시 활동을 중단하고 동아리 회장에게 보고한다."},
        {"doc_id": "S5", "kind": "규정", "text": "외부 팀과의 친선경기는 부장 승인 후 최소 1주일 전에 공지해야 한다."},
        {"doc_id": "S6", "kind": "규정", "text": "훈련 불참 시 24시간 전에 단체 채팅방에 사전 공지를 남긴다."},
질문:
가장 중요한 규정 하나 골라줘

JSON 형식:
{json.dumps(case_schema, ensure_ascii=False, indent=2)}
'''

print(case_structured_prompt)

In [ ]:
case_model_answers = {
    "Gemini": '''{
"doc_id": "",
"reason": "제공된 문서 묶음에는 각 규정의 중요도나 우선순위를 판단할 수 있는 정량적·정성적 기준이 명시되어 있지 않으므로, 문서에 없는 내용을 임의로 추측하여 가장 중요한 규정 하나를 고를 수 없습니다.",
"missing_info": "각 규정 간의 우선순위, 위반 시 불이익, 또는 '가장 중요하다'고 판단할 수 있는 평가 기준에 대한 정보가 누락되어 있습니다."
}''',
    "ChatGPT": '''{
"selected_doc_id": "S4",
"selected_rule": "부상 발생 시 즉시 활동을 중단하고 동아리 회장에게 보고한다."
}
''',
}

pd.DataFrame([
    {
        "model": name,
        "char_len": len(text.strip()),
        "evidence_ids": ", ".join(sorted(set(re.findall(r"[A-Z]\d+", text)))),
    }
    for name, text in case_model_answers.items()
])

#### 추가 예제 실습 메모
- 내가 고른 예제:
- 어떤 LLM 2개를 비교했는가:
- 차이가 가장 컸던 부분:
- 내가 더 믿을 수 있다고 판단한 결과와 이유:

---
## 섹션 7. 질문 유형 바꿔 보기

같은 문서 묶음이라도 질문을 바꾸면 결과가 달라집니다.

In [ ]:
prompt_bank = [
    "가장 적합한 후보 1개를 고르고 이유를 설명해줘.",
    "후보를 순위대로 1위~3위까지 정리해줘.",
    "장점 / 위험 / 추가 필요 정보로 표를 만들어줘.",
    "가장 과신하면 안 되는 부분 2개를 짚어줘.",
    "의사결정 전에 꼭 더 확인해야 할 정보 3개를 적어줘.",
    "초보자도 바로 이해할 수 있는 체크리스트로 바꿔줘.",
]

pd.DataFrame({"extra_prompts": prompt_bank})

#### 질문 변형 메모
- 어떤 질문이 가장 좋은 답을 만들었는가:
- 어떤 질문은 너무 넓어서 오히려 답이 흐려졌는가:

---
## 섹션 8. 오늘의 정리

### 오늘 배운 것 3줄 정리
1.
2.
3.

### AI 사용 내역 1줄
- 사용 도구:
- 핵심 프롬프트:
- 내가 직접 검증한 것:

### 최종 비교 메모
- 가장 믿을 수 있었던 모델:
- 가장 다시 쓰기 편했던 출력 형식:
- 다음에 내 프로젝트에 가져갈 기술 1개:

### 참고 자료 및 추가 학습

*   **공식 문서:**
    *   [Colab 문서](https://colab.research.google.com/notebooks/intro.ipynb)
    *   [Google Gemini API 문서](https://ai.google.dev/)

*   **관련 튜토리얼:**
    *   [Python Pandas 시작하기](https://pandas.pydata.org/pandas-docs/stable/getting_started/install.html)
    *   [Matplotlib 튜토리얼](https://matplotlib.org/stable/tutorials/index.html)

*   **유용한 라이브러리:**
    *   `pandas`: 데이터 분석 및 조작
    *   `numpy`: 수치 계산
    *   `matplotlib`, `seaborn`: 데이터 시각화